# FraudShield AI — Credit Card Fraud Detection & Predictive Risk Analysis

**CA Predictive Analysis Project**

## Project objective
Build and evaluate machine-learning models that identify potentially fraudulent credit-card transactions. Because fraud is rare, the project emphasizes **precision, recall, F1-score, PR-AUC, confusion matrices and threshold selection**, not accuracy alone.

### Dataset
Place the supplied `creditcard.csv` in `../data/` relative to this notebook (or update `DATA_PATH` below). The optional `creditcard_clean.csv` can be used for a data-quality comparison only. The primary analysis starts from the original file to make preprocessing transparent.

### Important limitations
- `Class = 1` means fraud; `Class = 0` means legitimate.
- `V1`–`V28` are anonymized PCA-derived features; their original business meanings are not available.
- This is a historical, anonymized dataset. Model output is a risk flag, not proof of fraud or a real banking decision.
- Never use the test set to tune thresholds or choose models.

## 1. Import libraries and set project paths

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, precision_recall_curve,
    roc_curve
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

# If running this notebook from the notebooks folder, this is the expected location.
PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(PROJECT_DIR, "data", "creditcard.csv")
CLEAN_DATA_PATH = os.path.join(PROJECT_DIR, "data", "creditcard_clean.csv")

# Alternative: set DATA_PATH to the full path of your downloaded CSV.
print("Project directory:", PROJECT_DIR)
print("Dataset path:", DATA_PATH)

## 2. Load the original dataset

In [ ]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Cannot find {DATA_PATH}. Put creditcard.csv in the project's data folder "
        "or update DATA_PATH in this cell."
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
display(df.info())

## 3. Data audit and quality checks

In [ ]:
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("\\nClass counts:")
display(df["Class"].value_counts().rename(index={0:"Legitimate", 1:"Fraud"}))
print("\\nClass percentages:")
display((df["Class"].value_counts(normalize=True) * 100).round(4))
print("\\nDescriptive statistics:")
display(df[["Time", "Amount", "Class"]].describe().T)

### Remove exact duplicate transactions

The original file contains exact duplicate rows. We remove exact duplicates before splitting so repeated rows do not appear across training and test partitions and inflate evaluation results. We keep the original feature scale; scaling, where needed, is handled inside the model pipeline.

In [ ]:
rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {rows_before - len(df):,} exact duplicate rows.")
print("Shape after duplicate removal:", df.shape)
print("Class counts after duplicate removal:")
display(df["Class"].value_counts().rename(index={0:"Legitimate", 1:"Fraud"}))

### Optional: compare the supplied cleaned file
The uploaded cleaned file has standardized `Time` and `Amount` values and has had duplicate rows removed. It also has fewer fraud examples than the original file. We therefore do **not** mix it into the primary training data. This comparison helps document the difference between the two supplied files.

In [ ]:
if os.path.exists(CLEAN_DATA_PATH):
    clean_df = pd.read_csv(CLEAN_DATA_PATH)
    print("Original shape:", df.shape, "| Cleaned shape:", clean_df.shape)
    print("Original duplicate rows:", int(df.duplicated().sum()))
    print("Cleaned duplicate rows:", int(clean_df.duplicated().sum()))
    print("Original class counts:\\n", df["Class"].value_counts().to_dict())
    print("Cleaned class counts:\\n", clean_df["Class"].value_counts().to_dict())
    print("\\nOriginal Time / Amount summary:")
    display(df[["Time", "Amount"]].describe().T)
    print("\\nCleaned Time / Amount summary:")
    display(clean_df[["Time", "Amount"]].describe().T)
else:
    print("Optional cleaned file not found. Skipping comparison.")

## 4. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="Class")
plt.xticks([0,1], ["Legitimate (0)", "Fraud (1)"])
plt.title("Transaction class distribution")
plt.xlabel("Transaction class")
plt.ylabel("Number of transactions")
plt.tight_layout()
plt.show()

fraud_pct = df["Class"].mean() * 100
print(f"Fraud transactions: {int(df['Class'].sum()):,} ({fraud_pct:.4f}%)")
print(f"Legitimate transactions: {(df['Class']==0).sum():,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,4))
sns.histplot(data=df, x="Amount", hue="Class", bins=60, element="step",
             stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Transaction amount distribution")
axes[0].set_xlim(0, df["Amount"].quantile(0.99))

sns.boxplot(data=df, x="Class", y="Amount", showfliers=False, ax=axes[1])
axes[1].set_xticks([0,1], ["Legitimate", "Fraud"])
axes[1].set_title("Transaction amount by class")
plt.tight_layout()
plt.show()

In [ ]:
# Time is recorded as seconds elapsed from the first transaction in the dataset.
df["Hour_approx"] = ((df["Time"] // 3600) % 24).astype(int)
hourly = df.groupby(["Hour_approx", "Class"]).size().unstack(fill_value=0)
hourly.plot(kind="line", figsize=(11,4), marker="o")
plt.title("Transaction counts by approximate hour")
plt.xlabel("Approximate hour")
plt.ylabel("Transaction count")
plt.tight_layout()
plt.show()

## 5. Prepare features and target
We remove only the target from the feature matrix. The engineered `Hour_approx` feature is derived from `Time`. `Time` and `Amount` are scaled inside the logistic-regression pipeline; tree models do not require feature scaling.

We split the data **stratified** into training, validation and test partitions. The test set is kept aside until final evaluation.

In [ ]:
model_df = df.copy()
model_df["Hour_approx"] = ((model_df["Time"] // 3600) % 24).astype(int)

X = model_df.drop(columns=["Class"])
y = model_df["Class"].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "Fraud:", int(y_train.sum()))
print("Validation:", X_valid.shape, "Fraud:", int(y_valid.sum()))
print("Test:", X_test.shape, "Fraud:", int(y_test.sum()))

## 6. Train and compare baseline models

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE
        ))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=8, min_samples_leaf=5,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250, class_weight="balanced_subsample",
        min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE
    )
}

fitted_models = {}
validation_rows = []

for name, model in models.items():
    print("Training:", name)
    model.fit(X_train, y_train)
    fitted_models[name] = model

    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X_valid)[:, 1]
    else:
        scores = model.decision_function(X_valid)

    preds = (scores >= 0.5).astype(int)
    validation_rows.append({
        "Model": name,
        "Precision": precision_score(y_valid, preds, zero_division=0),
        "Recall": recall_score(y_valid, preds, zero_division=0),
        "F1": f1_score(y_valid, preds, zero_division=0),
        "ROC_AUC": roc_auc_score(y_valid, scores),
        "PR_AUC": average_precision_score(y_valid, scores)
    })

validation_results = pd.DataFrame(validation_rows).sort_values("PR_AUC", ascending=False)
display(validation_results)

## 7. Select a decision threshold using validation data
The default threshold of 0.50 is not necessarily best for fraud detection. We inspect the precision-recall trade-off on the **validation set only**. Choose a threshold based on the project's stated objective and explain the trade-off in your report.

In [ ]:
best_name = validation_results.iloc[0]["Model"]
best_model = fitted_models[best_name]
valid_scores = best_model.predict_proba(X_valid)[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_valid, valid_scores)
f1_values = (2 * precisions[:-1] * recalls[:-1]) / (
    precisions[:-1] + recalls[:-1] + 1e-12
)
best_idx = int(np.argmax(f1_values))
chosen_threshold = float(thresholds[best_idx])

print("Selected model by validation PR-AUC:", best_name)
print("Validation threshold maximizing F1:", round(chosen_threshold, 4))
print("Validation precision:", round(float(precisions[best_idx]), 4))
print("Validation recall:", round(float(recalls[best_idx]), 4))

plt.figure(figsize=(7,4))
plt.plot(thresholds, precisions[:-1], label="Precision")
plt.plot(thresholds, recalls[:-1], label="Recall")
plt.plot(thresholds, f1_values, label="F1")
plt.axvline(chosen_threshold, color="black", linestyle="--", label="Chosen threshold")
plt.xlabel("Decision threshold")
plt.ylabel("Score")
plt.title("Validation precision, recall and F1 vs threshold")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Final evaluation on the untouched test set

In [ ]:
test_scores = best_model.predict_proba(X_test)[:, 1]
test_preds = (test_scores >= chosen_threshold).astype(int)

print("Final model:", best_name)
print("Decision threshold:", round(chosen_threshold, 4))
print("\\nClassification report:")
print(classification_report(y_test, test_preds, target_names=["Legitimate", "Fraud"], zero_division=0))

final_metrics = {
    "Precision": precision_score(y_test, test_preds, zero_division=0),
    "Recall": recall_score(y_test, test_preds, zero_division=0),
    "F1": f1_score(y_test, test_preds, zero_division=0),
    "ROC_AUC": roc_auc_score(y_test, test_scores),
    "PR_AUC": average_precision_score(y_test, test_scores)
}
display(pd.DataFrame([final_metrics]).round(4))

In [ ]:
cm = confusion_matrix(y_test, test_preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted Legitimate", "Predicted Fraud"],
            yticklabels=["Actual Legitimate", "Actual Fraud"])
plt.title("Test-set confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_scores)
plt.figure(figsize=(6,4))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_test, test_scores):.4f}")
plt.plot([0,1], [0,1], linestyle="--", color="gray")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve — test set")
plt.legend()
plt.tight_layout()
plt.show()

p, r, _ = precision_recall_curve(y_test, test_scores)
plt.figure(figsize=(6,4))
plt.plot(r, p, label=f"PR-AUC = {average_precision_score(y_test, test_scores):.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-recall curve — test set")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Feature importance (tree-based model)

In [ ]:
rf_model = fitted_models["Random Forest"]
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(9,6))
sns.barplot(x=top_features.values, y=top_features.index)
plt.title("Top 15 Random Forest feature importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

display(top_features.to_frame("Importance"))

## 10. Test a single transaction
The example below uses an existing test-row as a demonstration. For a genuinely new transaction, collect all feature values using the same feature definitions and preprocessing pipeline.

In [ ]:
sample_transaction = X_test.iloc[[0]]
sample_score = float(best_model.predict_proba(sample_transaction)[:, 1][0])
sample_label = int(sample_score >= chosen_threshold)

print(f"Predicted fraud risk score: {sample_score:.4%}")
print("Model flag:", "POTENTIAL FRAUD — REVIEW" if sample_label == 1 else "LIKELY LEGITIMATE")
print("Reminder: this is a model flag, not proof of fraud.")

## 11. Save the trained model and evaluation outputs
The model is saved locally for reproducibility. Keep the threshold alongside the model.

In [ ]:
import joblib

os.makedirs(os.path.join(PROJECT_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, "outputs"), exist_ok=True)

artifact = {
    "model": best_model,
    "threshold": chosen_threshold,
    "model_name": best_name,
    "feature_columns": list(X.columns)
}
joblib.dump(artifact, os.path.join(PROJECT_DIR, "models", "fraudshield_model.joblib"))
validation_results.to_csv(os.path.join(PROJECT_DIR, "outputs", "validation_model_comparison.csv"), index=False)
pd.DataFrame([final_metrics]).to_csv(os.path.join(PROJECT_DIR, "outputs", "test_metrics.csv"), index=False)

print("Saved model and metrics in models/ and outputs/.")

## 12. Conclusion and business recommendations

Complete this section after running the notebook and replace the prompts with your actual results:

1. Best model: **[insert model]**
2. Test PR-AUC: **[insert value]**
3. Test precision: **[insert value]**
4. Test recall: **[insert value]**
5. Selected threshold: **[insert value]**
6. Main limitation: **[discuss anonymized features, historical data and false positives]**
7. Business recommendation: use the model to prioritize transactions for human review, with a threshold selected according to the cost of missed fraud versus unnecessary reviews.

### Viva points
- Why is accuracy misleading for this dataset?
- What is class imbalance?
- Why use stratified splitting?
- What is the difference between precision and recall?
- Why is PR-AUC useful for rare fraud cases?
- Why must oversampling be applied only to training data?
- Why is a model prediction not proof of fraud?